In [7]:
# %% [markdown]
# # Generating Watershed Division CSV from Parquet Network Data
#
# This notebook implements Algorithm 2 to segment a river network into divisions
# based on Strahler stream order and specified order thresholds. The basin network
# data, including upstream connectivity via 'us1' and 'us2' columns, is loaded
# from a Parquet file. The output is a CSV file where each link
# in the network is assigned a division ID for different stream order thresholds.
#
# **Algorithm 2 Core Logic (Initiated from Highest Order Links):**
# 1.  **Traversal:** For each stream order threshold, an upstream Breadth-First Search (BFS)
#     is initiated from link(s) having the **maximum stream order** in the dataset.
# 2.  **Initial Division:** Each of these highest-order links starts a new division ID.
# 3.  **Propagation & Splitting (from current link L to its upstream U):**
#     * If `order(U) != order(L)` AND `order(U) >= threshold`: U starts a new division.
#     * If `order(U) == order(L)` OR `order(U) < threshold`: U inherits the division ID from L.

# %% [code]
import pandas as pd
import numpy as np
from collections import defaultdict, deque
from pathlib import Path # For checking file existence
import requests # Needed for fetching data from URL (for plotting, if re-enabled)
import matplotlib.pyplot as plt # For plotting section (if re-enabled)
from matplotlib.patches import Patch
from cartopy.crs import PlateCarree
from cartopy.feature import BORDERS, COASTLINE, STATES
import matplotlib.patheffects as pe
import time # To add delays between API calls if needed


# %% [markdown]
# ## 1. Configuration
# Define input/output file paths and processing parameters.

# %% [code]
# --- Configuration ---
# IMPORTANT: Adjust this path to your actual Parquet file location
BASIN_NETWORK_PARQUET_FILE = '../hlm_data/structure/sangamon.gzip'
OUTPUT_CSV_FILE = 'watershed_division_algo2_highest_order_root.csv'
ORDER_THRESHOLDS = range(4, 9) # Generate subw_4 through subw_8

# %% [markdown]
# ## 2. Data Loading and Preprocessing
#
# Load basin network data from Parquet and prepare necessary lookup structures.
# Outlet detection is now based on the highest stream order.

# %% [code]
# --- Load Basin Network Data from Parquet ---
basin_df = pd.DataFrame() # Initialize empty DataFrame
try:
    print(f"Loading basin network data from Parquet: {BASIN_NETWORK_PARQUET_FILE}")
    if not Path(BASIN_NETWORK_PARQUET_FILE).is_file():
        raise FileNotFoundError(f"Parquet file not found at {BASIN_NETWORK_PARQUET_FILE}")

    basin_df = pd.read_parquet(BASIN_NETWORK_PARQUET_FILE)
    print(f"Successfully loaded {len(basin_df)} links from Parquet.")
    print(f"Index name: {basin_df.index.name}") 
    print(f"Columns found: {basin_df.columns.tolist()}")

    required_data_cols = ['us1', 'us2', 'strmOrder']
    if not all(col in basin_df.columns for col in required_data_cols):
        missing = [col for col in required_data_cols if col not in basin_df.columns]
        raise ValueError(f"Missing required data columns: {missing}. Ensure 'us1', 'us2', 'strmOrder' exist.")

    for col in ['us1', 'us2']:
        basin_df[col] = pd.to_numeric(basin_df[col], errors='coerce').fillna(0).astype(int)
        basin_df[col] = basin_df[col].replace(-1, 0)
        
    basin_df['strmOrder'] = pd.to_numeric(basin_df['strmOrder'], errors='coerce').fillna(0).astype(int)
    basin_df.index = pd.to_numeric(basin_df.index, errors='coerce').fillna(0).astype(int)

except FileNotFoundError as fnf_error: print(f"ERROR: {fnf_error}")
except ValueError as ve: print(f"ERROR: Data validation error - {ve}")
except Exception as e: print(f"An unexpected error occurred loading basin data: {type(e).__name__} - {e}")

outlets = [] # Initialize outlets list

if basin_df.empty:
    print("Cannot proceed: Basin data is empty or failed to load.")
else:
    print("Basin data sample (from Parquet):")
    print(basin_df.head())

    # --- Preprocessing: Build Network Lookups using 'us1' and 'us2' ---
    print("\nPreprocessing network data...")
    link_orders = pd.Series(basin_df['strmOrder'].values, index=basin_df.index).to_dict()
    parents_of = {} 
    all_link_ids_in_df = set(basin_df.index.unique()) # Use this for checking validity of us1/us2

    for link_idx, row in basin_df.iterrows():
        us1_val, us2_val = row.us1, row.us2
        # Ensure parents are valid links present in the dataset, otherwise treat as no parent (0)
        p1 = us1_val if us1_val in all_link_ids_in_df else 0
        p2 = us2_val if us2_val in all_link_ids_in_df else 0
        parents_of[link_idx] = (p1, p2)
            
    # --- MODIFIED Outlet Identification: Use Highest Stream Order Links ---
    if not link_orders: # Should not happen if basin_df loaded
        print("CRITICAL WARNING: link_orders dictionary is empty. Cannot determine outlets.")
    else:
        max_order = 0
        if link_orders: # Check if link_orders is not empty
            max_order = max(link_orders.values() or [0]) # Get max order, default to 0 if empty
        
        if max_order > 0 :
            outlets = [link_id for link_id, order_val in link_orders.items() if order_val == max_order]
            print(f"Identified {len(outlets)} outlet(s) based on highest stream order ({max_order}). Examples: {outlets[:5]}")
        else:
            print("Warning: Max stream order is 0 or no links found. Attempting fallback outlet identification.")
            # Fallback if max_order is 0 (e.g., all strmOrder are 0 or invalid)
            # This fallback tries to find links not listed as parents, effectively structural outlets
            all_parent_ids_from_us_cols = set()
            for us1_val, us2_val in parents_of.values():
                if us1_val != 0: all_parent_ids_from_us_cols.add(us1_val)
                if us2_val != 0: all_parent_ids_from_us_cols.add(us2_val)
            
            potential_outlets = list(all_link_ids_in_df - all_parent_ids_from_us_cols)
            if potential_outlets:
                outlets = potential_outlets
                print(f"Fallback: Identified {len(outlets)} potential structural outlets. E.g., {outlets[:5]}")
            else:
                print("CRITICAL WARNING: Could not identify any outlets even with fallbacks.")


    if not outlets and not basin_df.empty:
        print("CRITICAL WARNING: No outlets identified. Division generation will likely fail or be empty.")
    elif not basin_df.empty:
        print(f"Preprocessing complete. Using {len(outlets)} identified outlet(s) to start BFS.")

# %% [markdown]
# ## 3. Algorithm 2: Division Calculation using Upstream BFS
#
# This section implements the core logic for Algorithm 2, starting BFS from the highest order link(s).

# %% [code]
if not basin_df.empty and outlets: # Proceed only if data loaded and outlets found
    results_all_thresholds = {}
    master_link_list = sorted(list(all_link_ids_in_df))

    for threshold_order in ORDER_THRESHOLDS:
        print(f"\nProcessing for Stream Order Threshold >= {threshold_order}...")
        link_to_division_id = {} 
        division_id_counter = 0
        
        bfs_queue = deque()
        
        for outlet_link in outlets: # Now 'outlets' are the highest order links
            if outlet_link not in link_to_division_id:
                division_id_counter += 1
                link_to_division_id[outlet_link] = division_id_counter
                bfs_queue.append(outlet_link)

        processed_in_bfs = set(bfs_queue)

        while bfs_queue:
            current_L = bfs_queue.popleft()
            div_L = link_to_division_id[current_L]
            order_L = link_orders.get(current_L, 0)
            
            us1_of_L, us2_of_L = parents_of.get(current_L, (0, 0))
            
            for U_link in [us1_of_L, us2_of_L]:
                if U_link == 0 or U_link not in link_orders: continue
                if U_link in link_to_division_id: continue

                order_U = link_orders.get(U_link, 0)
                assigned_div_for_U = -1

                if order_U != order_L and order_U >= threshold_order:
                    division_id_counter += 1
                    assigned_div_for_U = division_id_counter
                elif order_U == order_L or order_U < threshold_order:
                    assigned_div_for_U = div_L
                else: 
                    assigned_div_for_U = div_L
                
                link_to_division_id[U_link] = assigned_div_for_U
                if U_link not in processed_in_bfs:
                     bfs_queue.append(U_link)
                     processed_in_bfs.add(U_link)

        results_all_thresholds[f'subw_{threshold_order}'] = [link_to_division_id.get(link, pd.NA) for link in master_link_list]

    output_df = pd.DataFrame({'LINKNO': master_link_list})
    for threshold in ORDER_THRESHOLDS:
        col_name = f'subw_{threshold}'
        output_df[col_name] = results_all_thresholds[col_name]

    output_df = output_df.fillna(0)
    for threshold in ORDER_THRESHOLDS:
         output_df[f'subw_{threshold}'] = output_df[f'subw_{threshold}'].astype(int)

    print("\n--- Sample of Generated Division Data (Algorithm 2, Highest Order Root) ---")
    print(output_df.head(10))
    
    if 5 in ORDER_THRESHOLDS and f'subw_5' in output_df:
        print("\nDivision counts for subw_5 (excluding 0):")
        counts_subw5 = output_df[output_df[f'subw_5'] > 0][f'subw_5'].value_counts().sort_index()
        if not counts_subw5.empty: print(counts_subw5)
        else: print(f"No links assigned to numbered divisions for subw_5.")
elif basin_df.empty:
    print("Skipping division calculation because basin_df is empty.")
else: # outlets list is empty but basin_df is not
    print("Skipping division calculation because no outlets were identified to start BFS.")


# %% [markdown]
# ## 4. Save to CSV

# %% [code]
if not basin_df.empty and 'output_df' in locals() and not output_df.empty:
    try:
        output_df.to_csv(OUTPUT_CSV_FILE, index=False)
        print(f"\nSuccessfully saved watershed divisions to: {OUTPUT_CSV_FILE}")
    except Exception as e:
        print(f"Error saving CSV file: {e}")
else:
    print("\nSkipping CSV save: Basin data not loaded or output_df not generated.")

Loading basin network data from Parquet: ../hlm_data/structure/sangamon.gzip
Successfully loaded 116779 links from Parquet.
Index name: LINKNO
Columns found: ['DSLINKNO', 'USLINKNO1', 'USLINKNO2', 'DSNODEID', 'strmOrder', 'Length', 'Magnitude', 'DSContArea', 'strmDrop', 'Slope', 'StraightL', 'USContArea', 'WSNO', 'DOUTEND', 'DOUTSTART', 'DOUTMID', 'np', 'us1', 'us2', 'us3', 'us4', 'us5', 'us6', 'us7', 'us8', 'us9', 'us10']
Basin data sample (from Parquet):
        DSLINKNO  USLINKNO1  USLINKNO2  DSNODEID  strmOrder  Length  \
LINKNO                                                                
308545    308769     308321      18465        -1          9   634.4   
308321    308545     308097     286594        -1          9    72.6   
18465     308545         -1         -1        -1          1   611.9   
308097    308321     307873     124417        -1          9   471.8   
286594    308321     283234     229698        -1          4   441.7   

        Magnitude    DSContArea  strmDrop

/tmp/ipykernel_4700/4138676408.py:191: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  output_df = output_df.fillna(0)
